<a href="https://colab.research.google.com/github/javisagredo-dev/Evaluacion1_Prog_DS/blob/main/Evaluaci%C3%B3n_1_Prog_DS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import geopandas as gpd
from google.colab import drive
from shapely.geometry import Point

In [8]:
# Montar Google Drive
drive.mount('/content/drive')

# ============================================
# 1. Cargar sismos
# ============================================
df = pd.read_csv(
    '/content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_1/Raw/archive.zip',
    compression='zip'
)

print(f"Total de sismos antes de filtrar: {len(df):,}")
#print(f"Columnas: {df.columns.tolist()}")
df.head(3)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total de sismos antes de filtrar: 1,054,683
Columnas: ['id', 'time', 'year', 'month', 'day_of_year', 'hour', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'place', 'type', 'tsunami', 'mag_category', 'depth_category', 'nst', 'gap', 'dmin', 'rms', 'net', 'updated', 'status']


,id,time,year,month,day_of_year,hour,latitude,longitude,depth,mag,...,tsunami,mag_category,depth_category,nst,gap,dmin,rms,net,updated,status
0,cent19000105190000000,1900-01-05 19:00:00+00:00,1900,1,5,19,-3.0,102.0,NaN,7.0,...,0,Major (6-7),NaN,NaN,NaN,NaN,NaN,cent,2025-04-18T00:16:56.005Z,reviewed
1,cent19000111090700000,1900-01-11 09:07:00+00:00,1900,1,11,9,-5.0,148.0,NaN,7.0,...,0,Major (6-7),NaN,NaN,NaN,NaN,NaN,cent,2025-04-18T22:55:15.621Z,reviewed
2,cent19000120063300000,1900-01-20 06:33:00+00:00,1900,1,20,6,20.0,-105.0,NaN,7.3,...,0,Great (7-8),NaN,NaN,NaN,NaN,NaN,cent,2025-04-19T23:36:34.400Z,reviewed


In [14]:
# ============================================
# 2. Cargar polígonos de Chile (tierra y mar)
# ============================================
print("\nCargando polígonos...")

# Tierra: Regiones de Chile
tierra = gpd.read_file(
    '/content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_1/Raw/REGIONES_v1.shp'
)

# Mar: ZEE de Chile (desde el XML)
mar = gpd.read_file(
    '/content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_1/Raw/eez.xml'
)

# ============================================
# CORRECCIÓN: Asignar CRS si no tiene
# ============================================
if tierra.crs is None:
    print("Asignando CRS a REGIONES (EPSG:4326)...")
    tierra = tierra.set_crs("EPSG:4326")

if mar.crs is None:
    print("Asignando CRS a ZEE (EPSG:4326)...")
    mar = mar.set_crs("EPSG:4326")

# Asegurar mismo sistema de coordenadas (WGS84 = EPSG:4326)
tierra = tierra.to_crs("EPSG:4326")
mar = mar.to_crs("EPSG:4326")

print(f"CRS de tierra: {tierra.crs}")
print(f"CRS de mar: {mar.crs}")

# Unir tierra + mar en un solo polígono
chile_total = pd.concat([tierra, mar], ignore_index=True)
chile_total = gpd.GeoDataFrame(chile_total, geometry='geometry', crs="EPSG:4326")
chile_total = chile_total.dissolve()  # Fusiona todo en un solo polígono

print("Polígonos cargados y unificados correctamente")



Cargando polígonos...
Asignando CRS a REGIONES (EPSG:4326)...
Asignando CRS a ZEE (EPSG:4326)...
CRS de tierra: EPSG:4326
CRS de mar: EPSG:4326
Polígonos cargados y unificados correctamente


In [15]:
# ============================================
# 3. Convertir sismos a GeoDataFrame
# ============================================
print("\nCreando geometría de puntos...")

sismos_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
    crs="EPSG:4326"
)

print(f"{len(sismos_gdf):,} puntos creados")


Creando geometría de puntos...
1,054,683 puntos creados


In [16]:
# ============================================
# 4. Filtrar sismos dentro de Chile
# ============================================
print("\nFiltrando sismos dentro de Chile...")

# Usamos intersects() en lugar de within() para incluir bordes
sismos_chile = sismos_gdf[sismos_gdf.intersects(chile_total.geometry.iloc[0])]

print(f"\n{'='*50}")
print(f"RESULTADOS DEL FILTRADO:")
print(f" Sismos TOTALES: {len(df):,}")
print(f" Sismos en CHILE: {len(sismos_chile):,}")
print(f" Sismos FUERA: {len(df) - len(sismos_chile):,}")
print(f" Porcentaje en Chile: {(len(sismos_chile)/len(df))*100:.2f}%")
print(f"{'='*50}")


Filtrando sismos dentro de Chile...

RESULTADOS DEL FILTRADO:
 Sismos TOTALES: 1,054,683
 Sismos en CHILE: 39,616
 Sismos FUERA: 1,015,067
 Porcentaje en Chile: 3.76%


In [17]:
sismos_chile.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 39616 entries, 83 to 1054663
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   id              39616 non-null  object  
 1   time            39616 non-null  object  
 2   year            39616 non-null  int64   
 3   month           39616 non-null  int64   
 4   day_of_year     39616 non-null  int64   
 5   hour            39616 non-null  int64   
 6   latitude        39616 non-null  float64 
 7   longitude       39616 non-null  float64 
 8   depth           39616 non-null  float64 
 9   mag             39616 non-null  float64 
 10  magType         39616 non-null  object  
 11  place           39616 non-null  object  
 12  type            39616 non-null  object  
 13  tsunami         39616 non-null  int64   
 14  mag_category    39616 non-null  object  
 15  depth_category  39616 non-null  object  
 16  nst             20744 non-null  float64 
 17  gap   

In [ ]:
import folium
from folium.plugins import HeatMap

# Asegurarte de que no haya nulos en lat/lon
sismos_chile = sismos_chile.dropna(subset=['latitude', 'longitude'])

# Crear mapa centrado en Chile
mapa_chile = folium.Map(
    location=[-35.5, -71.0],
    zoom_start=4,
    tiles='CartoDB positron'
)

# Extraer coordenadas para el heatmap
heat_data = sismos_chile[['latitude', 'longitude']].values.tolist()

# Agregar mapa de calor
HeatMap(
    heat_data,
    radius=8,
    blur=6,
    max_zoom=10
).add_to(mapa_chile)

# Mostrar mapa
mapa_chile

In [23]:
##EXPORTACION DEL DATAFRAME A PROCESSED
import os

ruta_output = "/content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_1/Processed/sismos_chile.csv"
sismos_chile.to_csv(ruta_output, index=False)

print(f"Archivo CSV guardado en: {ruta_output}")

Archivo CSV guardado en: /content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_1/Processed/sismos_chile.csv


In [26]:
## REVISION
sismos_chile.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 39616 entries, 83 to 1054663
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   id              39616 non-null  object  
 1   time            39616 non-null  object  
 2   year            39616 non-null  int64   
 3   month           39616 non-null  int64   
 4   day_of_year     39616 non-null  int64   
 5   hour            39616 non-null  int64   
 6   latitude        39616 non-null  float64 
 7   longitude       39616 non-null  float64 
 8   depth           39616 non-null  float64 
 9   mag             39616 non-null  float64 
 10  magType         39616 non-null  object  
 11  place           39616 non-null  object  
 12  type            39616 non-null  object  
 13  tsunami         39616 non-null  int64   
 14  mag_category    39616 non-null  object  
 15  depth_category  39616 non-null  object  
 16  nst             20744 non-null  float64 
 17  gap   

In [28]:
tipos_mag = sismos_chile['magType'].unique()

# Mostrar la lista
print("Tipos de magType encontrados:")
print(tipos_mag)

Tipos de magType encontrados:
['ms' 'mw' 'mb' 'ml' 'md' 'mwb' 'mwc' 'm' 'mww' 'mwr']


In [29]:
sismos_chile.describe()

,year,month,day_of_year,hour,latitude,longitude,depth,mag,tsunami,nst,gap,dmin,rms
count,39616.000000,39616.000000,39616.000000,39616.000000,39616.000000,39616.000000,39616.000000,39616.000000,39616.0,20744.000000,21970.000000,7989.000000,20840.000000
mean,2004.057553,6.397819,179.038621,11.251918,-30.652409,-71.177511,52.228406,3.969833,0.0,25.244360,156.732685,0.741086,0.848651
std,11.786147,3.391142,103.707701,6.942842,5.677600,1.424574,40.692926,0.778974,0.0,45.492986,71.545027,0.620833,0.340679
min,1904.000000,1.000000,1.000000,0.000000,-57.543300,-81.035600,0.000000,2.500000,0.0,4.000000,12.500000,0.004000,0.000000
25%,1998.000000,3.000000,88.000000,5.000000,-33.913000,-71.910000,20.000000,3.400000,0.0,7.000000,101.625000,0.387000,0.600000
50%,2004.000000,6.000000,175.000000,11.000000,-32.398000,-71.344000,35.000000,4.000000,0.0,12.000000,147.100000,0.632000,0.860000
75%,2011.000000,9.000000,268.000000,17.000000,-28.039250,-70.376000,85.809000,4.400000,0.0,23.000000,207.000000,0.893000,1.100000
max,2026.000000,12.000000,366.000000,23.000000,-17.561000,-67.028200,435.000000,9.500000,0.0,714.000000,351.700000,10.422000,3.880000
